In [1]:
import json
import glob
import os
from pathlib import Path
import time
import os
from dotenv import load_dotenv

load_dotenv()

def consolidar_jsons(fonte, cidade, PASTA_DADOS):
    
    now = time.strftime("%Y-%m")
    
    padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.json')
    
    arquivos_json = glob.glob(padrao_busca)
    
    if not arquivos_json:
        print("Nenhum arquivo encontrado com o padrão especificado.")
        return

    dados_consolidados = []
    total_arquivos = len(arquivos_json)

    print(f"Iniciando a união de {total_arquivos} arquivos...")

    for i, caminho in enumerate(arquivos_json, 1):
        nome_base = os.path.basename(caminho)
        with open(caminho, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                # Verifica se o conteúdo é uma lista (padrão do seu scraper)
                if isinstance(conteudo, list):
                    dados_consolidados.extend(conteudo)
                else:
                    dados_consolidados.append(conteudo)
                
                print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(conteudo)} itens)")
            except Exception as e:
                print(f"Erro ao ler {nome_base}: {e}")

    # 2. Salva o arquivo final consolidado
    nome_final = f'{cidade}_{fonte}_{now}.json'
    
    caminho_final = PASTA_DADOS / nome_final

    try:
        with open(caminho_final, 'w', encoding='utf-8') as f_out:
            json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        """if tamanho_final > 0 and len(dados_consolidados) > 0:
            print(f"✅ Consolidação concluída: {len(dados_consolidados)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos anteriores apenas se o final estiver OK
           
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_json:
                try:
                    os.remove(arquivo_velho)
                    print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                except Exception as e:
                    print(f"   Erro ao excluir {arquivo_velho}: {e}")
        
        
            
            print("✨ Limpeza concluída com sucesso!")
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")"""

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo consolidado: {e}")


In [5]:

import asyncio
import sys
from pathlib import Path
import time

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / 'balneario_camboriu'

#consolidar_jsons('olx', 'balneario_camboriu', PASTA_DADOS)


In [6]:
PASTA_DADOS

WindowsPath('c:/Users/jefer/Documents/Ciencia-de-dados/Preco-Imoveis/dados/balneario_camboriu')

In [2]:
import pandas as pd
import warnings
import logging
import asyncio
from funcoes_limpando_dados_imoveis import (limpar_valor_iptu,
                                            limpar_banheiros, 
                                            limpar_metragem, 
                                            limpar_vagas,  
                                            #limpa_endereco_apply, 
                                            limpar_valor_condominio, 
                                            converter_para_data, 
                                            classificar_tipo_imovel, 
                                            reclassificar_outros, 
                                            preencher_todas_coordenadas,
                                            main_example, 
                                            limpar_valor_venda, 
                                            limpar_quartos, 
                                            pirabeiraba_dona_francisca, 
                                            geocodificar_dataframe,
                                            limpa_endereco_apply_zap, 
                                            limpa_endereco_apply_chave_mao, 
                                            limpa_endereco_apply_olx,)
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time

cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / cidade

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore")

start_time = time.time()

async def limpando_dados_cidades(pd_data, batch, cidade_limpeza = 'joinville', estado_limpeza = 'sc', cidade_localizacao = 'Joinville', estado_localizacao = 'SC',  tipo_async=True,  pais='Brasil'): 
       
    logger.info("Iniciando o processo de limpeza de dados de imóveis...")    
    
    pd_data = pd_data.drop_duplicates(subset=['url'])

    pd_data = pd_data[pd_data['valor_imovel'].notna()]

    pd_data_sem_nulos = pd_data.dropna(thresh=10)

    logger.info(f"Removendo linhas com muitos valores faltantes. Registros restantes: {pd_data_sem_nulos.shape}")

    #endereco_dividido = pd_data_sem_nulos['endereco'].apply(limpa_endereco_apply)

    #pd_data_endereco_dividido = pd.concat([pd_data_sem_nulos, endereco_dividido], axis=1).drop('endereco', axis=1)

    #pd_data_endereco_dividido['bairro'] = pd_data_endereco_dividido['bairro'].apply(pirabeiraba_dona_francisca)

    #logger.info("Coluna 'endereco' dividida em 'rua', 'bairro', 'cidade' e 'estado'...")
    
    pd_data_estado = pd_data_sem_nulos.copy()

    pd_data_estado = pd_data_estado[pd_data_estado['estado'] == estado_limpeza]

    logger.info(f"Removendo linhas com estado diferente de {estado_limpeza}. Registros restantes: {pd_data_estado.shape}")

    pd_data_metragem = pd_data_estado.copy()

    pd_data_metragem['metragem'] = pd_data_metragem['metragem'].apply(limpar_metragem)

    logger.info(f"Coluna 'metragem' limpa. Registros restantes: {pd_data_metragem.shape}")

    pd_data_valor_imovel = pd_data_metragem.copy()

    try:
        pd_data_valor_imovel['valor_venda'] = pd_data_valor_imovel['valor_venda'].apply(limpar_valor_venda)
    except:
        pd_data_valor_imovel['valor_imovel'] = pd_data_valor_imovel['valor_imovel'].apply(limpar_valor_venda)


    logger.info(f"Coluna 'valor_venda' limpa. Registros restantes: {pd_data_valor_imovel.shape}")

    pd_data_valor_condominio = pd_data_valor_imovel.copy()

    pd_data_valor_condominio['condominio'] = pd_data_valor_condominio['condominio'].apply(limpar_valor_condominio)

    logger.info(f"Coluna 'condominio' limpa. Registros restantes: {pd_data_valor_condominio.shape}")

    pd_data_valor_iptu = pd_data_valor_condominio.copy()

    pd_data_valor_iptu['iptu'] = pd_data_valor_iptu['iptu'].apply(limpar_valor_iptu)
    

    logger.info(f"Coluna 'iptu' limpa. Registros restantes: {pd_data_valor_iptu.shape}")

    pd_data_ano_publicacao = pd_data_valor_iptu.copy()

    pd_data_ano_publicacao['data_criacao'] = pd_data_ano_publicacao['data_criacao'].apply(converter_para_data)

    pd_data_ano_publicacao['dias_publicacao'] = (pd.to_datetime(datetime.now().strftime('%Y-%m-%d')) - pd.to_datetime(pd_data_ano_publicacao['data_criacao'], format='%d/%m/%Y')).dt.days
    
    logger.info(f"Coluna 'data_criacao' limpa. Registros restantes: {pd_data_ano_publicacao.shape}")

    pd_data_banheiros = pd_data_ano_publicacao.copy()
    
    pd_data_banheiros['banheiros'] = pd_data_banheiros['banheiros'].apply(limpar_banheiros)

    logger.info(f"Coluna 'banheiros' limpa. Registros restantes: {pd_data_banheiros.shape}")

    pd_data_quartos = pd_data_banheiros.copy()

    pd_data_quartos['quartos'] = pd_data_quartos['quartos'].apply(limpar_quartos)

    logger.info(f"Coluna 'quartos' limpa. Registros restantes: {pd_data_quartos.shape}")

    pd_data_garagem = pd_data_quartos.copy()

    #pd_data_garagem['vagas'] = pd_data_garagem['vagas'].replace('--', 0).astype('int64')

    pd_data_garagem['vagas'] = pd_data_garagem['vagas'].apply(limpar_vagas)

    logger.info(f"Coluna 'vagas' limpa. Registros restantes: {pd_data_garagem.shape}")

    pd_data_tipo_imovel = pd_data_garagem.copy()

    pd_data_tipo_imovel['tipo_imovel'] = pd_data_tipo_imovel['titulo'].apply(classificar_tipo_imovel)

    mask = pd_data_tipo_imovel['tipo_imovel'] == 'outros'

    pd_data_tipo_imovel.loc[mask, 'tipo_imovel'] = (
        pd_data_tipo_imovel.loc[mask, 'descricao']
        .apply(reclassificar_outros)
    )

    logger.info(f"Coluna 'tipo_imovel' classificada. Registros restantes: {pd_data_tipo_imovel.shape}")

    pd_data_long_lat = pd_data_tipo_imovel.copy()
    
    if tipo_async:
        pd_data_lat_log_completo = await preencher_todas_coordenadas(pd_data_long_lat, batch_size=batch, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)
    else:
        pd_data_lat_log_completo = geocodificar_dataframe(pd_data_long_lat, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)

    logger.info(f"Todas as coordenadas preenchidas. Registros restantes: {pd_data_lat_log_completo.shape}")

    pd_data_lat_log_completo['preco_por_m2'] = pd_data_lat_log_completo['valor_imovel'] / pd_data_lat_log_completo['metragem']

    logger.info(f'Coluna "preco_por_m2" criada. Registros restantes: {pd_data_lat_log_completo.shape}')

    def classificar_dentro_bairro(grupo):
        p25 = grupo["preco_por_m2"].quantile(0.25)
        p50 = grupo["preco_por_m2"].quantile(0.50)
        p75 = grupo["preco_por_m2"].quantile(0.75)
        
        def faixa(val):
            if val <= p25:
                return "barato"
            elif val <= p50:
                return "medio_baixo"
            if val <= p75:
                return "medio_alto"
            else:
                return "alto_padrao"

        grupo = grupo.copy()
        grupo["faixa"]       = grupo["preco_por_m2"].apply(faixa)
        grupo["p25_bairro"]  = p25
        grupo["p50_bairro"]  = p50
        grupo["p75_bairro"]  = p75
        return grupo

    pd_data_range_bairro_tipo_imovel = pd_data_lat_log_completo.groupby(["bairro", "tipo_imovel"], group_keys=False, ).apply(classificar_dentro_bairro)

    pd_data_range_bairro_tipo_imovel = pd.concat([pd_data_lat_log_completo, pd_data_range_bairro_tipo_imovel[['faixa', 'p25_bairro', 'p50_bairro', 'p75_bairro']]], axis=1)

    logger.info(f"Criando Faixas de preço por bairro e tipo de imóvel classificadas. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")

    pd_data_range_bairro_tipo_imovel["desvio_mediana"] = round((pd_data_range_bairro_tipo_imovel["preco_por_m2"] - pd_data_range_bairro_tipo_imovel["p50_bairro"]) / pd_data_range_bairro_tipo_imovel["p50_bairro"],2)

    logger.info(f"Coluna 'desvio_mediana' criada. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")
    
    return pd_data_range_bairro_tipo_imovel

In [3]:
def carregar_json(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo mais recente pelo padrão e retorna um DataFrame.
    Retorna DataFrame vazio se não encontrar nenhum arquivo.
    """
    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    
    logger.info(f"Arquivo encontrado: {arquivo.name}")

    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return pd.DataFrame(data), arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")
        
def carregar_parquet(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo .parquet mais recente pelo padrão e retorna um DataFrame.
    """
    # Garante que estamos buscando arquivos .parquet se o pattern não especificar
    if not glob_pattern.endswith('.parquet'):
        glob_pattern = glob_pattern.replace('.json', '.parquet')

    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    # Ordena para pegar o mais recente (mantendo sua lógica de data no final do nome)
    try:
        arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    except Exception:
        arquivo = max(arquivos, key=lambda f: f.stat().st_mtime) # Fallback para data de modificação
    
    logger.info(f"Arquivo Parquet encontrado: {arquivo.name}")

    try:
        # No Parquet, o pandas lê o arquivo diretamente pelo caminho
        df = pd.read_parquet(arquivo)
        return df, arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    # Esta função permanece igual, pois Path.unlink() deleta qualquer tipo de arquivo
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")

In [4]:
def normalizar_bairros(bairro, mapeamento):
    if not isinstance(bairro, str):
        return bairro
        
    bairro_low = bairro.lower()
    
    for nome_correto, variacoes in mapeamento.items():
        # Verifica se qualquer uma das variações está contida no nome original
        if any(v in bairro_low for v in variacoes):
            return nome_correto
            
    return bairro 

async def limpando_dados(name_arquivo_zap : str, 
         name_arquivo_vivareal: str, 
         name_arquivo_chave_mao: str,
         name_arquivo_olx: str,
         name_arquivo_saida: str,
         pasta_dados : Path, 
         batch: int = 1,
         tipo_async: bool = False,
         cidade_localizacao: str = 'Joinville', 
         cidade_limpeza: str = 'joinville',
         estado_limpeza: str = 'sc', 
         estado_localizacao: str = 'SC',
         pais: str = 'Brasil', 
         MAPA_BAIRROS: dict = None,):
    
    logger.info(f"Iniciando limpeza de dados de imóveis de {cidade_limpeza}...")
    
    logger.info(f"Pasta de dados: {pasta_dados}")

    pasta_dados.mkdir(parents=True, exist_ok=True)

    df_zap, arquivo_zap      = carregar_json(pasta_dados, name_arquivo_zap)
    
    df_vivareal, arquivo_vivareal = carregar_json(pasta_dados,name_arquivo_vivareal)
    
    df_chave_mao, arquivo_chave_mao = carregar_json(pasta_dados,name_arquivo_chave_mao)
    
    df_olx, arquivo_olx = carregar_json(pasta_dados,name_arquivo_olx)

    if not df_zap.empty:
        df_zap['fonte'] = 'zap_imoveis'
    if not df_vivareal.empty:
        df_vivareal['fonte'] = 'viva_real'

    if not df_chave_mao.empty:
        df_chave_mao['fonte'] = 'chave_mao'
    
    if not df_olx.empty:
        df_olx['fonte'] = 'olx'

    if df_zap.empty and df_vivareal.empty and df_chave_mao.empty and df_olx.empty:
        logger.error("Nenhum dado encontrado em nenhuma das fontes — abortando.")
        return
    
    #df_zap_endereco_limpo = df_zap['endereco'].apply(limpa_endereco_apply_zap)
    #df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(limpa_endereco_apply_zap)
    #df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(limpa_endereco_apply_chave_mao)
    #df_olx_endereco_limpo = df_olx['endereco'].apply(limpa_endereco_apply_olx)
    
    df_zap_endereco_limpo = df_zap['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(lambda x: limpa_endereco_apply_chave_mao(x, cidade_limpeza, estado_limpeza))
    df_olx_endereco_limpo = df_olx['endereco'].apply(lambda x: limpa_endereco_apply_olx(x, cidade_limpeza, estado_limpeza))
    
    df_zap_endereco =  pd.concat([df_zap, df_zap_endereco_limpo], axis=1)
    df_vivareal_endereco =  pd.concat([df_vivareal, df_vivareal_endereco_limpo], axis=1)
    df_chave_mao_endereco =  pd.concat([df_chave_mao, df_chave_mao_endereco_limpo], axis=1)
    df_olx_endereco =  pd.concat([df_olx, df_olx_endereco_limpo], axis=1)

    df = pd.concat([df_zap_endereco if not df_zap_endereco.empty else pd.DataFrame(), 
                    df_vivareal_endereco if not df_vivareal_endereco.empty else pd.DataFrame(),
                    df_chave_mao_endereco if not df_chave_mao_endereco.empty else pd.DataFrame(),
                    df_olx_endereco if not df_olx_endereco.empty else pd.DataFrame()], 
                   axis=0, ignore_index=True)
    
    logger.info(f"Total de registros carregados: {len(df)} (zap: {len(df_zap)} | vivareal: {len(df_vivareal)} | chave_mao: {len(df_chave_mao)} | olx: {len(df_olx)})")

    # Limpeza
    df_limpo = await limpando_dados_cidades(df, 
                                        batch = batch, 
                                        cidade_limpeza= cidade_limpeza, 
                                        cidade_localizacao= cidade_localizacao,
                                        tipo_async=tipo_async,
                                        estado_limpeza= estado_limpeza,
                                        estado_localizacao= estado_localizacao, 
                                        pais= pais
                                        )

    # Remove duplicatas
    colunas_dedup = ['valor_imovel', 'rua', 'bairro', 'metragem', 'quartos', 'preco_por_m2', 'banheiros', 'lat', 'lng']
    
    colunas_dedup = [c for c in colunas_dedup if c in df_limpo.columns]  

    antes = len(df_limpo)
    
    df_limpo = df_limpo.drop_duplicates(subset=colunas_dedup, keep='first').reset_index(drop=True)
    
    logger.info(f"Duplicatas removidas: {antes - len(df_limpo)} | Registros finais: {len(df_limpo)}")
    
    if MAPA_BAIRROS:
        df_limpo['bairro'] = df_limpo['bairro'].apply(normalizar_bairros, args=(MAPA_BAIRROS,))
    
    logger.info(f"Coluna 'bairro' corrigida...")
    
    #df_limpo = df_limpo.groupby('bairro').filter(lambda x: len(x) > 1)
    
    return df_limpo




In [17]:
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time
import pandas as pd

bairro = 'pinheiros'

cidade = f'sao_paulo/{bairro}'

estado = 'sc'

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / cidade

pd_data = pd.read_parquet(PASTA_DADOS / f'sao_paulo_{bairro}_imoveis_limpo_2026-05.parquet')



In [18]:
pd_data['bairro'].value_counts()

bairro
pinheiros               34876
alto de pinheiros         447
pacaembu                  338
jardim guedala            264
jardim paulista           244
                        ...  
jardim germania             2
vila cavaton                2
nucleo do engordador        2
jardim maristela            2
eldorado                    2
Name: count, Length: 319, dtype: int64

In [21]:
pd_data[pd_data['bairro'].isin(['pinheiros'])]['fonte'].value_counts()

fonte
olx            19058
chave_mao      15042
viva_real        511
zap_imoveis      265
Name: count, dtype: int64

In [19]:
pd_data['fonte'].value_counts()

fonte
olx            26750
chave_mao      15320
viva_real        513
zap_imoveis      266
Name: count, dtype: int64

In [24]:
pd_data['bairro'].value_counts().head(50)

bairro
pinheiros                34876
alto de pinheiros          447
pacaembu                   338
jardim guedala             264
jardim paulista            244
jardim dos estados         217
vila madalena              195
brooklin paulista          185
parque dos principes       182
higienopolis               181
fazenda morumbi            179
jardim america             175
jardim petropolis          172
cidade jardim              163
chacara santo antonio      163
alto da lapa               153
santo amaro                152
jardim leonor              149
city america               141
vila nova conceicao        133
campo belo                 126
jardim europa              125
sumare                     119
jardim cordeiro            119
morumbi                    111
jardim paulistano          100
vila ida                    94
itaim bibi                  93
butanta                     90
boacava                     86
cerqueira cesar             83
interlagos                  78
l

In [25]:
pd_data_limpo = pd_data[pd_data['bairro'].isin(['pinheiros'])]

In [26]:
pd_data_limpo

,url,titulo,metragem,banheiros,vagas,quartos,valor_imovel,condominio,endereco,iptu,...,dias_publicacao,tipo_imovel,lat,lng,preco_por_m2,faixa,p25_bairro,p50_bairro,p75_bairro,desvio_mediana
0,https://www.zapimoveis.com.br/imovel/venda-ter...,"Terreno / Lote Comercial à venda, 1m² - Pinheiros",1.0,0,0,0,4500000.0,0.0,"Pinheiros, São Paulo - SP",2000.0,...,511.0,terreno,-23.567249,-46.701951,4.500000e+06,alto_padrao,15000.000000,30113.077236,530000.000000,148.44
1,https://www.zapimoveis.com.br/imovel/venda-ter...,"Terreno / Lote Comercial à venda, 1m² - Pinheiros",1.0,0,0,0,5000000.0,0.0,"Rua Mourato Coelho, 1109 - Pinheiros, São Paul...",1998.0,...,221.0,terreno,-23.558340,-46.691318,5.000000e+06,alto_padrao,15000.000000,30113.077236,530000.000000,165.04
2,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 3 Quartos à venda, 10m² - Pinh...",10.0,1,0,3,8000000.0,NaN,"Rua Henrique Monteiro, 154 - Pinheiros, São Pa...",NaN,...,8.0,apartamento,-23.569271,-46.690278,8.000000e+05,alto_padrao,13333.333333,18054.166667,22571.428571,43.31
3,https://www.zapimoveis.com.br/imovel/venda-ter...,"Terreno / Lote Comercial à venda, 1m² - Pinheiros",1.0,0,0,0,3201000.0,0.0,"Rua Fidalga, 741 - Pinheiros, São Paulo - SP",1500.0,...,393.0,terreno,-23.553576,-46.691970,3.201000e+06,alto_padrao,15000.000000,30113.077236,530000.000000,105.30
4,https://www.zapimoveis.com.br/imovel/venda-pin...,"para venda ou aluguel, 10m² - Pinheiros",10.0,0,0,0,5300000.0,NaN,"Rua Teodoro Sampaio - Pinheiros, São Paulo - SP",3245.0,...,532.0,terreno,-23.563990,-46.688840,5.300000e+05,medio_alto,15000.000000,30113.077236,530000.000000,16.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42402,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Apartamento em Pinheiros com 2 dormitórios sen...,98.0,3,2,2,1850000.0,1100.0,"Pinheiros, São Paulo, SP, 05409002",800.0,...,NaN,apartamento,-23.554856,-46.677498,1.887755e+04,medio_alto,13333.333333,18054.166667,22571.428571,0.05
42403,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,15334 - Residential / Apartment 1 Dorm. (1 Suí...,100.0,2,2,1,1700000.0,1200.0,"Pinheiros, São Paulo, SP, 05414025",540.0,...,NaN,apartamento,-23.561368,-46.681289,1.700000e+04,medio_baixo,13333.333333,18054.166667,22571.428571,-0.06
42404,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Apartamento para venda e locação com 101m² em ...,101.0,1,0,3,1600000.0,1594.0,"Pinheiros, São Paulo, SP, 05412001",358.0,...,NaN,apartamento,-23.560840,-46.677394,1.584158e+04,medio_baixo,13333.333333,18054.166667,22571.428571,-0.12
42606,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Casa com 1 quartos à venda ou para locação em ...,1000.0,0,1,1,14000000.0,0.0,"Pinheiros, São Paulo, SP, 05408003",0.0,...,NaN,casa,-23.568273,-46.694657,1.400000e+04,medio_baixo,11250.000000,14737.500000,20000.000000,-0.05


In [27]:
pd_data_limpo.to_parquet(PASTA_DADOS / f'sao_paulo_{bairro}_imoveis_limpo_2026-05.parquet')